In [ ]:
import os
import cv2
import hashlib
import shutil
import numpy as np

# --- CẤU HÌNH CELL 1 ---
RAW_DIR = "raw_data"           # Thư mục chứa ảnh gốc lộn xộn
TEMP_DIR = "temp_clean_data"   # Thư mục tạm (chứa ảnh sạch nhưng chưa đổi tên)
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff'}

# Tạo thư mục tạm (Xóa cũ nếu có để làm mới)
if os.path.exists(TEMP_DIR):
    shutil.rmtree(TEMP_DIR)
os.makedirs(TEMP_DIR)

print(f"BẮT ĐẦU BƯỚC 1: LỌC RÁC & TRÙNG LẶP...")
print(f"reading from: {RAW_DIR} -> saving to: {TEMP_DIR}")

seen_hashes = set()
stats = {"total": 0, "corrupt": 0, "duplicate": 0, "kept": 0}

for root, dirs, files in os.walk(RAW_DIR):
    for filename in files:
        file_path = os.path.join(root, filename)
        ext = os.path.splitext(filename)[1].lower()
        if ext not in VALID_EXTS: continue
        stats["total"] += 1
        try:
            # 1. Kiểm tra File lỗi (Sanity Check)
            img = cv2.imread(file_path)
            if img is None:
                print(f"File lỗi (Corrupted): {filename}")
                stats["corrupt"] += 1
                continue
            img_hash_input = cv2.resize(img, (100, 100))
            img_hash_input = cv2.cvtColor(img_hash_input, cv2.COLOR_BGR2GRAY)
            file_hash = hashlib.md5(img_hash_input.tobytes()).hexdigest()

            if file_hash in seen_hashes:
                stats["duplicate"] += 1
                continue
            seen_hashes.add(file_hash)
            shutil.copy2(file_path, os.path.join(TEMP_DIR, filename))
            stats["kept"] += 1
            if stats["total"] % 500 == 0:
                print(f"⏳ Đã quét {stats['total']} file...")

        except Exception as e:
            print(f"⚠️ Lỗi lạ với {filename}: {e}")
            stats["corrupt"] += 1

Khởi tạo data

In [ ]:
import os
import cv2
INPUT_DIR = "temp_clean_data"  # Lấy đầu ra của Cell 1 làm đầu vào
OUTPUT_DIR = "final_dataset"   # Thư mục kết quả cuối cùng
IMG_SIZE_LIMIT = 1280

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

print(f"BẮT ĐẦU BƯỚC 2: CHUẨN HÓA & ĐỔI TÊN...")
print(f"reading from: {INPUT_DIR} -> saving to: {OUTPUT_DIR}")

count = 0
files = os.listdir(INPUT_DIR) # Lấy danh sách ảnh sạch

for filename in files:
    try:
        file_path = os.path.join(INPUT_DIR, filename)
        # Đọc ảnh
        img = cv2.imread(file_path)
        if img is None: continue # Phòng hờ
        # cv2.imwrite với đuôi .jpg sẽ tự động convert mọi định dạng (png, bmp...) về jpg
        cv2.imwrite(save_path, img, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
        count += 1
        if count % 100 == 0:
            print(f"Đã chuẩn hóa {count} ảnh...")

    except Exception as e:
        print(f"Lỗi khi xử lý {filename}: {e}")


ModuleNotFoundError: No module named 'roboflow'